# Autonomous Mower — YOLOv8n Object Detection

This notebook trains a **YOLOv8n** object detection model on your Roboflow obstacle dataset.

| | Details |
|---|---|
| Model | YOLOv8n (nano) |
| Parameters | 3.2M |
| Model size | ~6 MB |
| Classes | Person, Dog, Tree, Bicycle, Electric pole, Uncovered manhole |
| Dataset | 24,893 images (with 3x augmentation) |
| Jetson Nano FPS | ~30 FPS (TensorRT FP16) |

**Before running:** Runtime → Change runtime type → **A100 GPU**

In [ ]:
# Step 0: Install dependencies
!pip install -q ultralytics roboflow

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
# Step 1: Download dataset from Roboflow (already in YOLOv8 format)
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("malhanaf-s-workspace").project("obstacle-detection-yeuzf-f5w2d")
version = project.version(1)
dataset = version.download("yolov8")

DATASET_DIR = dataset.location
print(f"\nDataset at: {DATASET_DIR}")

In [ ]:
# Step 2: Inspect dataset
from pathlib import Path
import yaml

data_yaml = Path(DATASET_DIR) / "data.yaml"
with open(data_yaml) as f:
    data_config = yaml.safe_load(f)

print("data.yaml contents:")
print(f"  Classes: {data_config['names']}")
print(f"  NC: {data_config.get('nc', len(data_config['names']))}")

for split in ["train", "valid", "test"]:
    img_dir = Path(DATASET_DIR) / split / "images"
    if img_dir.exists():
        imgs = list(img_dir.glob("*"))
        print(f"  {split:6s}: {len(imgs)} images")
    else:
        print(f"  {split:6s}: not found")

In [ ]:
# Step 3: Train YOLOv8n
import os
from pathlib import Path
from ultralytics import YOLO

os.chdir("/content")

model = YOLO("yolov8n.pt")  # pretrained nano detection

results = model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/mower_runs",
    name="obstacle_det",
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
)

RUN_DIR = Path(results.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"

print(f"\n{'='*50}")
print(f"  RUN_DIR  = {RUN_DIR}")
print(f"  best.pt  = {BEST_PT}")
print(f"  exists?  = {BEST_PT.exists()}")
print(f"{'='*50}")

In [ ]:
# Step 4: View training curves
from pathlib import Path
from IPython.display import Image, display

results_img = RUN_DIR / "results.png"
if results_img.exists():
    display(Image(filename=str(results_img), width=900))
else:
    print(f"results.png not at {results_img}")
    for f in sorted(RUN_DIR.iterdir()):
        print(f"  {f.name}")

## Evaluation

- **Validation** = used during training to pick the best checkpoint
- **Test** = touched only once for the final unbiased score

Key metrics:
- **mAP50**: mean Average Precision at IoU=0.50
- **mAP50-95**: mean AP averaged over IoU 0.50→0.95 (stricter)

In [ ]:
# Step 5: Evaluate on VALIDATION set
from ultralytics import YOLO

best_model = YOLO(str(BEST_PT))

val_metrics = best_model.val(data=str(data_yaml), split="val", device=0)

print(f"\n{'='*40}")
print(f"  VALIDATION RESULTS")
print(f"{'='*40}")
print(f"  mAP50     : {val_metrics.box.map50:.4f}")
print(f"  mAP50-95  : {val_metrics.box.map:.4f}")

In [ ]:
# Step 6: Evaluate on TEST set (final unbiased score — report this)
test_metrics = best_model.val(data=str(data_yaml), split="test", device=0)

print(f"\n{'='*40}")
print(f"  FINAL TEST SET RESULTS")
print(f"{'='*40}")
print(f"  mAP50     : {test_metrics.box.map50:.4f}")
print(f"  mAP50-95  : {test_metrics.box.map:.4f}")
print(f"\n  Report these numbers in your capstone.")

In [ ]:
# Step 7: Visualize predictions on test images (report-ready)
import matplotlib.pyplot as plt
import cv2
import numpy as np

test_img_dir = Path(DATASET_DIR) / "test" / "images"
test_images = sorted(test_img_dir.glob("*"))[:8]

fig, axes = plt.subplots(len(test_images), 2, figsize=(16, 5 * len(test_images)))
if len(test_images) == 1:
    axes = [axes]

for i, img_path in enumerate(test_images):
    preds = best_model.predict(source=str(img_path), conf=0.25, device=0, verbose=False)
    result = preds[0]
    orig = cv2.cvtColor(result.orig_img, cv2.COLOR_BGR2RGB)

    # Annotated image with boxes and labels
    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)

    axes[i][0].imshow(orig)
    axes[i][0].set_title(f"Input: {img_path.name}", fontsize=11)
    axes[i][0].axis("off")

    axes[i][1].imshow(annotated)
    n_det = len(result.boxes) if result.boxes is not None else 0
    axes[i][1].set_title(f"Detections: {n_det} objects", fontsize=11)
    axes[i][1].axis("off")

plt.suptitle("YOLOv8n — Test Set Object Detection Results", fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig("detection_results.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved to detection_results.png")

In [ ]:
# Step 8: View confusion matrix
from IPython.display import Image, display

cm = RUN_DIR / "confusion_matrix_normalized.png"
if cm.exists():
    print("Confusion Matrix (normalized):")
    display(Image(filename=str(cm), width=700))
else:
    print("Confusion matrix not found")

In [ ]:
# Step 9: Export to ONNX for Jetson Nano deployment
onnx_path = best_model.export(format="onnx", imgsz=640, simplify=True)
print(f"ONNX saved to: {onnx_path}")
print(f"\nOn Jetson Nano, convert ONNX → TensorRT for maximum speed:")
print(f"  /usr/src/tensorrt/bin/trtexec --onnx=best.onnx --saveEngine=best.engine --fp16")

In [ ]:
# Step 10: Download results
from google.colab import files
import shutil

shutil.make_archive("mower_det_results", "zip", str(RUN_DIR))

files.download(str(BEST_PT))
files.download(str(onnx_path))
files.download("detection_results.png")
files.download("mower_det_results.zip")
if (RUN_DIR / "results.png").exists():
    files.download(str(RUN_DIR / "results.png"))

print("\nDownloaded files:")
print("  best.pt                   → YOLOv8n detection weights")
print("  best.onnx                 → ONNX export for Jetson")
print("  detection_results.png     → visual results for report")
print("  mower_det_results.zip     → full training run (curves, metrics, weights)")